# Deep Learning Fundamentals — Perceptron, Neural Networks and Propagation

## Exercise 1: Short Quiz — Deep Learning Concepts

**1. What is the key difference between traditional machine learning and deep learning?**

Traditional ML requires **manual feature engineering**: a human expert must decide which raw inputs (e.g., edges in an image, n-grams in text) to extract before feeding them to a model. Deep learning performs **automatic feature extraction**: a neural network with multiple layers learns hierarchical representations directly from raw data (pixels, audio waveforms, text tokens), eliminating the need for hand-crafted features.

---

**2. How do artificial neural networks (ANNs) mimic the human brain?**

ANNs are loosely inspired by biological neurons. Each **artificial neuron** receives inputs, multiplies them by learned weights (analogous to synaptic strengths), sums them, adds a bias, and passes the result through a non-linear activation function — similar to how a biological neuron fires when the combined input exceeds a threshold. Neurons are organised in **layers** (input → hidden → output), mirroring cortical processing, where lower layers detect simple features and higher layers combine them into abstract concepts.

---

**3. Why does deep learning perform better on large datasets compared to traditional ML?**

Deep learning models have **millions of parameters** and can learn complex non-linear mappings. With small datasets, this capacity leads to overfitting, so simpler ML models (SVM, Random Forest) often win. With large datasets, deep learning can fully exploit its representational power, continuously improving as more data is provided, whereas classical models tend to plateau. Put simply, more data provides a stronger gradient signal to train the many parameters of a deep network effectively.

---

**4. What are some challenges of deep learning, and how can they be addressed?**

| Challenge | Mitigation |
|---|---|
| Requires large labelled datasets | Transfer learning, data augmentation, semi-supervised learning |
| High computational cost | GPUs/TPUs, model pruning, knowledge distillation |
| Overfitting | Dropout, L1/L2 regularisation, early stopping, batch normalisation |
| Interpretability (black box) | Attention mechanisms, SHAP values, saliency maps |
| Vanishing/exploding gradients | ReLU activations, batch normalisation, gradient clipping, residual connections |
| Hyperparameter tuning complexity | Automated ML (AutoML), Bayesian optimisation |

---

**5. What is feature engineering, and why is it not needed in deep learning?**

Feature engineering is the process of manually transforming raw data into informative representations for a model (e.g., computing pixel gradients for image classification, or extracting bag-of-words for text). Deep learning eliminates this step because convolutional layers learn spatial filters, recurrent layers learn temporal patterns, and dense layers learn abstract combinations — the network discovers optimal representations automatically during training via backpropagation.

---

**6. What role do hidden layers play in a deep learning model?**

Hidden layers transform the input representation progressively into increasingly abstract features. In a digit classifier: the first hidden layer might detect edges, the second detects curves and corners, the third detects digit parts, and the output layer assembles these into class probabilities. Without hidden layers (a single linear layer), the model can only learn linearly separable patterns. **Depth** is what allows neural networks to model highly non-linear, compositional data.

---

**7. What is the function of an activation function in an ANN?**

An activation function introduces **non-linearity** into the network. Without it, stacking multiple layers would be equivalent to a single linear transformation (because a composition of linear functions is still linear). Common activation functions:
- **ReLU** `f(x) = max(0, x)` — fast to compute, avoids vanishing gradient for positive values.
- **Sigmoid** `f(x) = 1/(1+e⁻ˣ)` — squashes output to (0,1), used for binary classification outputs.
- **Softmax** — normalises a vector of scores into a probability distribution over classes, used in the output layer of multi-class classifiers.
- **Tanh** — zero-centred output in (−1,1), often preferred over sigmoid in hidden layers.

## Exercise 2: Simple Perceptron Decision System

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ── Perceptron parameters ────────────────────────────────────
w_temp = 0.6   # weight for Temperature
w_rain = 0.4   # weight for Rain
bias   = 2     # bias term
THRESHOLD = 20 # step activation threshold

def weighted_sum(temperature, rain):
    return (temperature * w_temp) + (rain * w_rain) + bias

def step_activation(z, threshold=THRESHOLD):
    return 1 if z > threshold else 0

def perceptron(temperature, rain):
    z      = weighted_sum(temperature, rain)
    output = step_activation(z)
    return z, output

# ── Case 1: Temperature = 70°F, Rain = 0 ─────────────────────
z1, out1 = perceptron(70, 0)
print('=== Case 1: Temperature = 70°F, Rain = No (0) ===')
print(f'  Weighted Sum = (70 × 0.6) + (0 × 0.4) + 2 = {z1:.1f}')
print(f'  Step output  = {out1}  →  {"Go outside ✓" if out1 else "Stay inside ✗"}')
print()

# ── Case 2: Temperature = 50°F, Rain = 1 ─────────────────────
z2, out2 = perceptron(50, 1)
print('=== Case 2: Temperature = 50°F, Rain = Yes (1) ===')
print(f'  Weighted Sum = (50 × 0.6) + (1 × 0.4) + 2 = {z2:.1f}')
print(f'  Step output  = {out2}  →  {"Go outside ✓" if out2 else "Stay inside ✗"}')

In [ ]:
# ── Visualisation: decision boundary over temperature and rain ─
temps = np.linspace(0, 80, 300)
rains = [0, 1]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, rain_val in zip(axes, [0, 1]):
    zs      = temps * w_temp + rain_val * w_rain + bias
    outputs = (zs > THRESHOLD).astype(int)

    ax.plot(temps, zs, color='steelblue', linewidth=2, label='Weighted Sum')
    ax.axhline(THRESHOLD, color='crimson', linestyle='--', linewidth=1.5,
               label=f'Threshold ({THRESHOLD})')
    ax.fill_between(temps, zs, THRESHOLD,
                    where=outputs==1, alpha=0.15, color='green', label='Go outside zone')
    ax.fill_between(temps, zs, THRESHOLD,
                    where=outputs==0, alpha=0.15, color='red', label='Stay inside zone')

    # Plot the two case points when relevant
    if rain_val == 0:
        ax.scatter([70], [z1], s=120, color='green', zorder=5, label=f'Case 1 (z={z1:.0f})')
    else:
        ax.scatter([50], [z2], s=120, color='red',   zorder=5, label=f'Case 2 (z={z2:.0f})')

    ax.set_title(f'Rain = {"Yes" if rain_val else "No"} — Perceptron Output',
                 fontsize=12, fontweight='bold')
    ax.set_xlabel('Temperature (°F)')
    ax.set_ylabel('Weighted Sum')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.suptitle('Perceptron Decision System — Go Outside?', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### Interpretation

**Case 1 (70°F, no rain):** Weighted sum = `70 × 0.6 + 0 × 0.4 + 2 = 44.0`. Since 44 > 20, the perceptron outputs **1 → Go outside**. The temperature is warm enough to outweigh the threshold easily.

**Case 2 (50°F, rain):** Weighted sum = `50 × 0.6 + 1 × 0.4 + 2 = 32.4`. Since 32.4 > 20, the perceptron also outputs **1 → Go outside**. While rain contributes negatively to comfort (weight 0.4 is smaller), the temperature of 50°F still pushes the weighted sum above the threshold of 20.

The temperature weight (0.6) dominates the decision. For the perceptron to suggest staying inside, the temperature would need to be below approximately 30°F (no rain) — a scenario neither case satisfies.

## Exercise 3: Neural Network for MNIST Digit Classification

In [ ]:
# Step 1: Import TensorFlow and Keras modules
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.datasets import mnist
from tensorflow.keras.utils import to_categorical

print('TensorFlow version:', tf.__version__)

In [ ]:
# Step 2: Load the MNIST dataset
(X_train, y_train), (X_test, y_test) = mnist.load_data()

print(f'Training images shape : {X_train.shape}  — {len(y_train)} labels')
print(f'Test images shape     : {X_test.shape}   — {len(y_test)} labels')
print(f'Pixel value range     : {X_train.min()} – {X_train.max()}')

In [ ]:
# Step 3: Normalize the data (scale pixel values to [0, 1])
X_train = X_train / 255.0
X_test  = X_test  / 255.0

print(f'Pixel value range after normalisation: {X_train.min():.1f} – {X_train.max():.1f}')

In [ ]:
# Step 4: One-hot encode the labels
y_train_ohe = to_categorical(y_train, num_classes=10)
y_test_ohe  = to_categorical(y_test,  num_classes=10)

print(f'Label before OHE: {y_train[0]}  →  After OHE: {y_train_ohe[0]}')

In [ ]:
# Step 5: Build the neural network model
model = models.Sequential([
    layers.Flatten(input_shape=(28, 28)),          # 28×28 → 784 vector
    layers.Dense(128, activation='relu'),          # hidden layer
    layers.Dense(10,  activation='softmax'),       # output layer (10 classes)
], name='MNIST_Classifier')

model.summary()

In [ ]:
# Step 6: Compile the model
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
print('Model compiled.')

In [ ]:
# Step 7: Train the model
history = model.fit(
    X_train, y_train_ohe,
    epochs=10,
    batch_size=128,
    validation_split=0.1,
    verbose=1
)

In [ ]:
# Evaluate on test set
test_loss, test_acc = model.evaluate(X_test, y_test_ohe, verbose=0)
print(f'Test Accuracy : {test_acc:.4f}  ({test_acc*100:.2f}%)')
print(f'Test Loss     : {test_loss:.4f}')

In [ ]:
# ── Training history plots ─────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].plot(history.history['accuracy'],     label='Train', color='#4C72B0', linewidth=2)
axes[0].plot(history.history['val_accuracy'], label='Val',   color='#E84040', linewidth=2)
axes[0].set_title('Model Accuracy over Epochs', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history.history['loss'],     label='Train', color='#4C72B0', linewidth=2)
axes[1].plot(history.history['val_loss'], label='Val',   color='#E84040', linewidth=2)
axes[1].set_title('Model Loss over Epochs', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('MNIST Training History', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Exercise 4: Forward Propagation — House Price Prediction

In [ ]:
import numpy as np

# ── Input values ───────────────────────────────────────────────
x1 = 2000   # Square Footage
x2 = 3      # Number of Bedrooms

# ── Weights and bias ──────────────────────────────────────────
w1 = 0.5    # Weight for Square Footage
w2 = 0.7    # Weight for Number of Bedrooms
b  = 50_000 # Bias (base price)

# ── Step 1: Compute the weighted sum z ────────────────────────
z = (x1 * w1) + (x2 * w2) + b

print('=== Forward Propagation — House Price ===')
print(f'  z = (x1 × w1) + (x2 × w2) + b')
print(f'  z = ({x1} × {w1}) + ({x2} × {w2}) + {b:,}')
print(f'  z = {x1*w1:,.1f} + {x2*w2:.1f} + {b:,}')
print(f'  z = {z:,.1f}')

# ── Step 2: Apply ReLU activation ────────────────────────────
def relu(z):
    return max(0, z)

price_pred = relu(z)

print()
print('=== ReLU Activation ===')
print(f'  ReLU(z) = max(0, {z:,.1f}) = {price_pred:,.1f}')
print()
print(f'Predicted House Price: ${price_pred:,.0f}')

In [ ]:
# ── Visualise how the prediction changes with square footage ───
sqft_range = np.arange(500, 5001, 100)
predictions = [relu((sf * w1) + (x2 * w2) + b) for sf in sqft_range]

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(sqft_range, predictions, color='steelblue', linewidth=2.5)
ax.scatter([x1], [price_pred], color='crimson', s=120, zorder=5,
           label=f'Our case: {x1} sqft → ${price_pred:,.0f}')
ax.set_title('Predicted House Price vs Square Footage (3 Bedrooms, ReLU)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Square Footage')
ax.set_ylabel('Predicted Price ($)')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f'${x/1e3:.0f}K'))
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Interpretation

- **z = 2000 × 0.5 + 3 × 0.7 + 50,000 = 51,002.1**
- **ReLU(51,002.1) = 51,002.1** (since z > 0, ReLU does not clip the value)
- **Predicted house price: $51,002**

The bias term (50,000) acts as the base price of a house regardless of size or bedrooms. The weights scale how much each additional square foot (0.5) and bedroom (0.7) contributes to the price. ReLU ensures the predicted price can never be negative — if the weighted sum were somehow negative (physically impossible in this setup), ReLU would clip it to 0.

## Exercise 5 (Optional): Forward and Backward Propagation in Python

In [ ]:
import numpy as np

# ── Input data ─────────────────────────────────────────────────
x = np.array([4, 80])   # 4 hours studied, previous test score = 80

# ── Initial weights and bias ───────────────────────────────────
w = np.array([0.6, 0.3])
b = 10

# ── Forward Propagation ───────────────────────────────────────
def forward_propagation(x, w, b):
    z = np.dot(x, w) + b  # weighted sum (linear activation for regression)
    return z

y_pred = forward_propagation(x, w, b)
y_true = 85   # actual exam score

# ── Loss: Mean Squared Error (MSE) ────────────────────────────
loss = 0.5 * (y_true - y_pred) ** 2

# ── Gradients (Backpropagation) ───────────────────────────────
grad_w = -(y_true - y_pred) * x   # ∂Loss/∂w
grad_b = -(y_true - y_pred)       # ∂Loss/∂b

# ── Weight update (Gradient Descent) ─────────────────────────
learning_rate = 0.01
w_new = w - learning_rate * grad_w
b_new = b - learning_rate * grad_b

print('=== Initial Forward Pass ===')
print(f'  Prediction (y_pred) : {y_pred:.4f}')
print(f'  True value (y_true) : {y_true}')
print(f'  Error (y_true-y_pred): {y_true - y_pred:.4f}')
print(f'  Loss (0.5 × error²) : {loss:.4f}')
print()
print('=== Gradients ===')
print(f'  grad_w = {grad_w}')
print(f'  grad_b = {grad_b:.4f}')
print()
print('=== Updated Parameters ===')
print(f'  w (before) = {w}  →  w (after) = {w_new.round(6)}')
print(f'  b (before) = {b}  →  b (after) = {b_new:.6f}')

In [ ]:
# ── Run multiple gradient descent steps and track loss ─────────
def train(x, y_true, w_init, b_init, learning_rate=0.01, epochs=100):
    w, b    = w_init.copy().astype(float), float(b_init)
    history = {'loss': [], 'pred': [], 'w0': [], 'w1': [], 'b': []}
    for _ in range(epochs):
        y_pred  = np.dot(x, w) + b
        loss    = 0.5 * (y_true - y_pred) ** 2
        grad_w  = -(y_true - y_pred) * x
        grad_b  = -(y_true - y_pred)
        w      -= learning_rate * grad_w
        b      -= learning_rate * grad_b
        history['loss'].append(loss)
        history['pred'].append(y_pred)
        history['w0'].append(w[0])
        history['w1'].append(w[1])
        history['b'].append(b)
    return w, b, history

w_final, b_final, hist = train(x, 85, np.array([0.6, 0.3]), 10, learning_rate=0.01, epochs=200)

print(f'Final prediction : {np.dot(x, w_final) + b_final:.4f} (target = 85)')
print(f'Final loss       : {hist["loss"][-1]:.6f}')
print(f'Final weights    : {w_final.round(6)}')
print(f'Final bias       : {b_final:.6f}')

In [ ]:
# ── Visualise convergence ───────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

axes[0].plot(hist['loss'], color='crimson', linewidth=2)
axes[0].set_title('Loss over Training Steps', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss (MSE)')
axes[0].grid(True, alpha=0.3)

axes[1].plot(hist['pred'], color='steelblue', linewidth=2)
axes[1].axhline(85, color='crimson', linestyle='--', linewidth=1.5, label='Target (85)')
axes[1].set_title('Prediction Convergence', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Predicted Score')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

axes[2].plot(hist['w0'], linewidth=2, label='w₀ (study hours)')
axes[2].plot(hist['w1'], linewidth=2, label='w₁ (prev score)')
axes[2].plot(hist['b'],  linewidth=2, linestyle='--', label='bias')
axes[2].set_title('Weight Evolution', fontsize=12, fontweight='bold')
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('Value')
axes[2].legend(fontsize=9)
axes[2].grid(True, alpha=0.3)

plt.suptitle('Gradient Descent — Backpropagation in Action', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Effect of different learning rates ─────────────────────────
lrs = [0.001, 0.01, 0.05, 0.1]

fig, ax = plt.subplots(figsize=(10, 5))
for lr in lrs:
    _, _, h = train(x, 85, np.array([0.6, 0.3]), 10, learning_rate=lr, epochs=200)
    ax.plot(h['loss'], linewidth=2, label=f'lr = {lr}')

ax.set_title('Loss vs Epochs for Different Learning Rates', fontsize=13, fontweight='bold')
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.legend()
ax.set_yscale('log')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Why Gradient Descent Reduces the Error

The gradient ∂Loss/∂w tells us the **direction and steepness** of the loss landscape with respect to each weight. Subtracting the gradient (scaled by the learning rate) moves the weights in the direction of **steepest descent** — i.e., toward a lower loss. Over many steps, this iteratively drives the prediction closer to the true target.

**Effect of learning rate:**
- Too small → slow convergence (many epochs needed)
- Too large → overshooting: the update jumps past the minimum and the loss may oscillate or diverge
- Optimal → loss converges smoothly and quickly to a minimum

## Exercise 6 (Optional): Visualizing MNIST Predictions

In [ ]:
# Step 0-6: Build + train (reuse model from Exercise 3 if already trained)
# This cell is self-contained so the exercise can run independently.
import tensorflow as tf
from tensorflow.keras.datasets import mnist
from tensorflow.keras.utils import to_categorical
from tensorflow.keras import layers, models

(X_train_6, y_train_6), (X_test_6, y_test_6) = mnist.load_data()
X_train_6 = X_train_6 / 255.0
X_test_6  = X_test_6  / 255.0

y_train_6_ohe = to_categorical(y_train_6, 10)
y_test_6_ohe  = to_categorical(y_test_6,  10)

model_6 = models.Sequential([
    layers.Flatten(input_shape=(28, 28)),
    layers.Dense(128, activation='relu'),
    layers.Dense(10,  activation='softmax'),
])
model_6.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model_6.fit(X_train_6, y_train_6_ohe, epochs=5, batch_size=128,
            validation_split=0.1, verbose=1)

# Step 7: Make predictions on the test set
y_proba_6  = model_6.predict(X_test_6, verbose=0)
y_pred_6   = y_proba_6.argmax(axis=1)

In [ ]:
# Step 8: Visualise predictions ─────────────────────────────────
import matplotlib.pyplot as plt
import numpy as np

# ── Grid of 25 sample predictions ─────────────────────────────
np.random.seed(42)
indices = np.random.choice(len(X_test_6), 25, replace=False)

fig, axes = plt.subplots(5, 5, figsize=(12, 12))
for ax, idx in zip(axes.flatten(), indices):
    ax.imshow(X_test_6[idx], cmap='gray')
    pred  = y_pred_6[idx]
    true  = y_test_6[idx]
    conf  = y_proba_6[idx][pred] * 100
    color = 'green' if pred == true else 'red'
    ax.set_title(f'Pred: {pred} ({conf:.0f}%)\nTrue: {true}',
                 fontsize=9, color=color, fontweight='bold')
    ax.axis('off')

plt.suptitle('MNIST Predictions (Green = Correct, Red = Incorrect)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Show misclassified examples with prediction confidence ─────
errors = np.where(y_pred_6 != y_test_6)[0]
print(f'Total misclassified: {len(errors)} / {len(X_test_6)} ({len(errors)/len(X_test_6)*100:.1f}%)')

fig, axes = plt.subplots(3, 5, figsize=(14, 9))
for ax, idx in zip(axes.flatten(), errors[:15]):
    ax.imshow(X_test_6[idx], cmap='gray')
    pred = y_pred_6[idx]
    true = y_test_6[idx]
    conf = y_proba_6[idx][pred] * 100
    ax.set_title(f'Pred: {pred} ({conf:.0f}%)\nTrue: {true}',
                 fontsize=9, color='crimson', fontweight='bold')
    ax.axis('off')

plt.suptitle('Misclassified MNIST Examples', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Prediction confidence histogram ───────────────────────────
correct_conf   = y_proba_6.max(axis=1)[y_pred_6 == y_test_6] * 100
incorrect_conf = y_proba_6.max(axis=1)[y_pred_6 != y_test_6] * 100

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(correct_conf,   bins=30, alpha=0.6, color='#55A868', edgecolor='white', label='Correct')
ax.hist(incorrect_conf, bins=30, alpha=0.6, color='#E84040', edgecolor='white', label='Incorrect')
ax.set_title('Model Confidence — Correct vs Incorrect Predictions', fontsize=13, fontweight='bold')
ax.set_xlabel('Confidence (%)')
ax.set_ylabel('Count')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'Mean confidence — correct    : {correct_conf.mean():.1f}%')
print(f'Mean confidence — incorrect  : {incorrect_conf.mean():.1f}%')
print('\nObservation: When the model is wrong, it is often still confident —')
print('these high-confidence errors are the most problematic to detect.')